# Two-Stage Hybrid Model Training with AutoGluon

**Goal**: Train two models for a hybrid forecasting approach:
1. **Classifier**: Predicts if a period is trading (`is_trading=1`) or not.
2. **Regressor**: Predicts `high`, `low`, `close`, `volume` for trading periods only, optimized for sMAPE.

## 1. Setup and Dependencies

In [1]:
import os
import pandas as pd
import numpy as np
import random
import time
import uuid
from tqdm.auto import tqdm
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import sys
import shutil
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

## 2. Configuration

In [2]:
logging.info("--- Setting up configuration ---")

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

BASE_DIR = project_root
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TRAIN_DIR_REG = os.path.join(DATA_DIR, "train_continuous")
VAL_DIR_REG = os.path.join(DATA_DIR, "val_continuous")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Define output paths for the two hybrid models
AUTOGLUON_CLASSIFIER_DIR = os.path.join(MODELS_DIR, "autogluon_trading_classifier")
AUTOGLUON_REGRESSOR_DIR = os.path.join(MODELS_DIR, "autogluon_hybrid_regressor")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_CLASSIFIER_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_REGRESSOR_DIR, exist_ok=True)

# Model configuration
TARGET_COLS = ["high", "low", "close", "volume"]
OUTPUT_CHUNK_LENGTH = 10
SEED = 827
TIME_LIMIT = 3600
MAX_NUMBER_FILES = 100
MAX_NUMBER_FILES_REG = 2500
MAX_EPOCH = 100

[INFO] --- Setting up configuration ---


## 3. Load Raw Data

In [3]:
def load_parquet_files(directory, file_list):
    """Load a specific list of parquet files from a directory."""
    data = {}
    for file in tqdm(file_list, desc=f"Loading {len(file_list)} files from {os.path.basename(directory)}"):
        asset_name = file.replace('.parquet', '')
        df = pd.read_parquet(os.path.join(directory, file))
        data[asset_name] = df
    return data

# Get common files and load them
train_files_clf = {f for f in os.listdir(TRAIN_DIR) if f.endswith('.parquet')}
val_files_clf = {f for f in os.listdir(VAL_DIR) if f.endswith('.parquet')}
common_files_clf = sorted(list(train_files_clf.intersection(val_files_clf)))

random.seed(SEED)
random.shuffle(common_files_clf)
files_to_load_clf = common_files_clf[:MAX_NUMBER_FILES]

raw_train_data_clf = load_parquet_files(TRAIN_DIR, files_to_load_clf)
raw_val_data_clf = load_parquet_files(VAL_DIR, files_to_load_clf)

# Get common files and load them
train_files_reg = {f for f in os.listdir(TRAIN_DIR_REG) if f.endswith('.parquet')}
val_files_reg = {f for f in os.listdir(VAL_DIR_REG) if f.endswith('.parquet')}
common_files_reg = sorted(list(train_files_reg.intersection(val_files_reg)))

random.seed(SEED)
random.shuffle(common_files_reg)
files_to_load_reg = common_files_reg[:MAX_NUMBER_FILES_REG]

raw_train_data_reg = load_parquet_files(TRAIN_DIR_REG, files_to_load_reg)
raw_val_data_reg = load_parquet_files(VAL_DIR_REG, files_to_load_reg)

# Define covariates for the regressor.
# 'is_trading' is excluded because it's always 1 in the filtered data.
PAST_COVARIATES = [
    'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1',
    'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2',
    'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3',
    'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4',
    'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5',
    'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6',
    'nearest_liquid_contract_close', 'cross_contract_mean',
]
FUTURE_COVARIATES= [
    'time_to_delivery', 'hour_of_day', 'day_of_week',
    'week_of_year', 'month', 'is_weekend',
]

Loading 100 files from train:   0%|          | 0/100 [00:00<?, ?it/s]

Loading 100 files from val:   0%|          | 0/100 [00:00<?, ?it/s]

Loading 2500 files from train_continuous:   0%|          | 0/2500 [00:00<?, ?it/s]

Loading 2500 files from val_continuous:   0%|          | 0/2500 [00:00<?, ?it/s]

## Part 1: Train Trading vs. Non-Trading Classifier

In [4]:
def convert_to_classifier_format(data_dict):
    """Converts data to be used for training the is_trading classifier."""
    all_data = []
    for asset_name, df in tqdm(data_dict.items(), desc="Converting for classifier"):
        df = df.copy()
        if len(df) < OUTPUT_CHUNK_LENGTH + 10:
            continue

        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)

        df.rename(columns={"ExecutionTime": "timestamp"}, inplace=True)
        df["item_id"] = asset_name
        all_data.append(df)

    combined_df = pd.concat(all_data, ignore_index=True)
    ts_df = TimeSeriesDataFrame.from_data_frame(
        combined_df,
        id_column='item_id',
        timestamp_column='timestamp'
    )
    return ts_df

logging.info("--- Preparing data for classifier ---")
train_ts_clf = convert_to_classifier_format(raw_train_data_clf)
val_ts_clf = convert_to_classifier_format(raw_val_data_clf)

[INFO] --- Preparing data for classifier ---


Converting for classifier:   0%|          | 0/100 [00:00<?, ?it/s]

Converting for classifier:   0%|          | 0/100 [00:00<?, ?it/s]

In [5]:
logging.info("--- Training classifier model ---")

# All other columns are treated as past covariates by default
classifier_predictor = TimeSeriesPredictor(
    prediction_length=OUTPUT_CHUNK_LENGTH,
    path=AUTOGLUON_CLASSIFIER_DIR,
    target="is_trading", # Target is 'is_trading'
    eval_metric="MASE", # Use a regression metric, it works for 0/1 targets
    freq="15min",
    known_covariates_names=FUTURE_COVARIATES,
    verbosity=2,
)

hyperparameters = {
    "SeasonalNaive": [
        {"max_epochs": 1}
    ],
    "TemporalFusionTransformer": [
        # zero-shot small
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        # fine-tune (standard)
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "lr": 1e-3, "batch_size": 64, "dropout": 0.1, "hidden_size": 128, "num_layers": 2, "patience": 10},
        # deeper variant (more capacity / longer training)
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "lr": 5e-4, "batch_size": 32, "dropout": 0.15, "hidden_size": 256, "num_layers": 3, "patience": 20}
    ],
    "PatchTST": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "patch_len": 16, "stride": 8, "d_model": 128, "n_heads": 4, "dropout": 0.1, "batch_size": 64, "lr": 1e-3},
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "patch_len": 32, "stride": 16, "d_model": 256, "n_heads": 8, "dropout": 0.15, "batch_size": 32, "lr": 5e-4}
    ],
    "DeepAR": [
        {"max_epochs": 1},
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "context_length": 96, "lr": 1e-3, "batch_size": 64, "num_layers": 2, "hidden_size": 128, "dropout": 0.1},
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "context_length": 128, "lr": 5e-4, "batch_size": 32, "num_layers": 3, "hidden_size": 256, "dropout": 0.15}
    ]
}


classifier_predictor.fit(
    train_data=train_ts_clf,
    hyperparameters=hyperparameters,
    time_limit=TIME_LIMIT,
    random_seed=SEED
)

logging.info("--- Evaluating classifier model ---")
classifier_leaderboard = classifier_predictor.leaderboard(val_ts_clf, silent=False)

[INFO] --- Training classifier model ---
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_trading_classifier'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          32
GPU Count:          1
Memory Avail:       14.61 GB / 31.63 GB (46.2%)
Disk Space Avail:   255.94 GB / 928.35 GB (27.6%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'freq': '15min',
 'hyperparameters': {'DeepAR': [{'max_epochs': 1},
                                {'ag_args': {'name_suffix': '_FineTuned'},
                                 'batch_size': 64,
                                 'context_length': 96,
                                 'dropout': 0.1,
                                 'hidden_size': 128,
          

                                     model  score_test  score_val  pred_time_test  pred_time_val  fit_time_marginal  fit_order
0                         WeightedEnsemble   -0.260722  -0.127260        7.698224      14.638101          12.830529         11
1                                   DeepAR   -0.260722  -0.127260        7.698224      14.638101          61.914774          5
2                      DeepAR_DeepFineTune   -0.388645  -0.176768        8.868598      14.208465         369.140644          7
3                         DeepAR_FineTuned   -0.415620  -0.192963        9.126641      14.343380         374.287902          6
4   TemporalFusionTransformer_DeepFineTune   -0.545441  -0.282058       29.323675      59.688769         360.086431          4
5      TemporalFusionTransformer_FineTuned   -0.559525  -0.285936       28.564731      59.667332         373.868325          3
6                    PatchTST_DeepFineTune   -0.582062  -0.323171        3.695370       4.237074         475.02

## Part 2: Train sMAPE Regressor on Trading Data

In [6]:
def convert_to_regressor_format(data_dict):
    """Convert dict of asset DataFrames to AutoGluon's TimeSeriesDataFrame format, FILTERING for trading periods."""
    all_data = []
    skipped_count = 0

    for asset_name, df in tqdm(data_dict.items(), desc="Converting for regressor"):
        df = df.copy()

        # CRITICAL: Filter for trading periods only for the regressor model
        if 'is_trading' in df.columns:
            df = df[df['is_trading'] == 1].copy()
            df.drop(columns=["is_trading"], inplace=True)

        if len(df) < OUTPUT_CHUNK_LENGTH + 10:
            skipped_count += 1
            continue

        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)

        df.rename(columns={"ExecutionTime": "timestamp"}, inplace=True)

        id_vars = [c for c in df.columns if c not in TARGET_COLS]
        melted_df = df.melt(
            id_vars=id_vars,
            value_vars=TARGET_COLS,
            var_name="target_col_name",
            value_name="target"
        )
        
        melted_df["item_id"] = asset_name + "_" + melted_df["target_col_name"]
        melted_df.drop(columns=["target_col_name"], inplace=True)
        all_data.append(melted_df)

    logging.info(f"Skipped {skipped_count} assets due to insufficient data after filtering for trading periods.")

    if not all_data:
        return None
        
    combined_df = pd.concat(all_data, ignore_index=True)
    ts_df = TimeSeriesDataFrame.from_data_frame(
        combined_df,
        id_column='item_id',
        timestamp_column='timestamp'
    )
    return ts_df

logging.info("--- Preparing data for regressor ---")
train_ts_reg = convert_to_regressor_format(raw_train_data_reg)
val_ts_reg = convert_to_regressor_format(raw_val_data_reg)

[INFO] --- Preparing data for regressor ---


Converting for regressor:   0%|          | 0/2500 [00:00<?, ?it/s]

[INFO] Skipped 0 assets due to insufficient data after filtering for trading periods.


Converting for regressor:   0%|          | 0/2500 [00:00<?, ?it/s]

[INFO] Skipped 0 assets due to insufficient data after filtering for trading periods.


In [7]:
logging.info("--- Training regressor model ---")

regressor_predictor = TimeSeriesPredictor(
    prediction_length=OUTPUT_CHUNK_LENGTH,
    path=AUTOGLUON_REGRESSOR_DIR,
    target="target",
    eval_metric="sMAPE", # Use sMAPE as required, now that zeros are filtered out
    freq="15min",
    known_covariates_names=FUTURE_COVARIATES,
    verbosity=2,
)

reg_hyperparameters = {
    "AutoETS": [
        {"max_epochs": 1}
    ],
    "TemporalFusionTransformer": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "lr": 1e-3, "batch_size": 64, "dropout": 0.1, "hidden_size": 128, "num_layers": 2, "patience": 10},
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "lr": 5e-4, "batch_size": 32, "dropout": 0.15, "hidden_size": 256, "num_layers": 3, "patience": 20}
    ],
    "PatchTST": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "patch_len": 16, "stride": 8, "d_model": 128, "n_heads": 4, "dropout": 0.1, "batch_size": 64, "lr": 1e-3},
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "patch_len": 32, "stride": 16, "d_model": 256, "n_heads": 8, "dropout": 0.15, "batch_size": 32, "lr": 5e-4}
    ],
    "DeepAR": [
        {"max_epochs": 1},
        {"max_epochs": MAX_EPOCH, "ag_args": {"name_suffix": "_FineTuned"},
         "context_length": 128, "lr": 1e-3, "batch_size": 64, "num_layers": 2, "hidden_size": 128, "dropout": 0.1},
        {"max_epochs": MAX_EPOCH * 2, "ag_args": {"name_suffix": "_DeepFineTune"},
         "context_length": 192, "lr": 5e-4, "batch_size": 32, "num_layers": 3, "hidden_size": 256, "dropout": 0.15}
    ],
    "DynamicOptimizedTheta": [
        {"max_epochs": 1}
    ]
}

regressor_predictor.fit(
    train_data=train_ts_reg,
    time_limit=TIME_LIMIT,
    hyperparameters=hyperparameters,
    random_seed=SEED
)

logging.info("--- Evaluating regressor model ---")
regressor_leaderboard = regressor_predictor.leaderboard(val_ts_reg, silent=False)

[INFO] --- Training regressor model ---
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_hybrid_regressor'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          32
GPU Count:          1
Memory Avail:       13.64 GB / 31.63 GB (43.1%)
Disk Space Avail:   255.48 GB / 928.35 GB (27.5%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': SMAPE,
 'freq': '15min',
 'hyperparameters': {'DeepAR': [{'max_epochs': 1},
                                {'ag_args': {'name_suffix': '_FineTuned'},
                                 'batch_size': 64,
                                 'context_length': 96,
                                 'dropout': 0.1,
                                 'hidden_size': 128,
            

                                     model  score_test  score_val  pred_time_test  pred_time_val  fit_time_marginal  fit_order
0                         WeightedEnsemble   -0.439514  -0.418500       43.997037      36.397782          32.722395         11
1      TemporalFusionTransformer_FineTuned   -0.439514  -0.418500       43.993542      36.397782         354.725327          3
2   TemporalFusionTransformer_DeepFineTune   -0.450627  -0.425901       54.302552      36.015844         354.087244          4
3                            SeasonalNaive   -0.461651  -0.456784       34.909042      31.443266           1.323457          1
4                                   DeepAR   -0.533349  -0.502902       71.536191      39.838017          13.892851          5
5   TemporalFusionTransformer_ZeroShotBase   -0.547909  -0.511655       44.371572      36.149387          45.023603          2
6                         DeepAR_FineTuned   -0.581188  -0.557729      132.353876     124.670100         399.63

## 5. Script Finished

In [8]:
logging.info("--- Hybrid model training finished successfully! ---")

[INFO] --- Hybrid model training finished successfully! ---
